## Import Modules

In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn import svm
from sklearn.datasets import make_blobs
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
import os
import glob

In [13]:
from modules.svm import svm_eval
from modules.knn import knn_eval
from modules.randomforest import rf_eval
from modules.logisticregression import lg_eval
from modules.xgboost import xgb_eval

## Load in Folder

In [15]:
# Load in your data here
folder_path = '..Data/Combination-Data-Files/Specific_DisordersPSD'
file_list = glob.glob(os.path.join(folder_path, '*.csv'))

## Model Training Loop

#### For each file inside the specified folder, the machine learning algorithms are run on all 6 bands. The accuracy and AUC of each are put into a csv file.

In [19]:
all_model_data = []
for file in file_list:
    basename = os.path.basename(file)  
    disorder_name = basename.split('_')[0]
    print(f'Processing file: {file} as disorder: {disorder_name}')  
    data = pd.read_csv(file)
    bands = [
        ('Delta', data.iloc[:, list(range(0,22))+list(range(118,289))+[-1]]),
        ('Theta', data.iloc[:, list(range(0,3))+list(range(22,41))+list(range(289,460))+[-1]]),
        ('Alpha', data.iloc[:, list(range(0,3)) + list(range(41,60))+list(range(460,631))+[-1]]),
        ('Beta', data.iloc[:, list(range(0,3)) + list(range(60,79))+list(range(631,802))+[-1]]),
        ('HighBeta', data.iloc[:, list(range(0,3)) + list(range(79,98))+list(range(802,973))+[-1]]),
        ('Gamma', data.iloc[:, list(range(0,3)) + list(range(98,117))+list(range(973,1144))+[-1]]),
        ('All', data.iloc[:, list(range(0,117))+list(range(118,1145))])
    ]

    for label, band in bands:
        # Data preprocessing
        print(f"\n{label} columns:\n", band.columns.tolist())
        X = band.drop('Class', axis=1)
        y = band['Class']
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.10, stratify=None)
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_train)
        pca = PCA(0.95)
        X = X.dropna()
        #X_pca = pca.fit_transform(X_train)
        #X_pca.shape
        X_train_pca = pca.fit_transform(X_scaled)
        #X_train_pca = pca.fit_transform(X_train)
        X_test_pca = pca.transform(X_test)
        #X_train_pca, X_test_pca, y_train, y_test = train_test_split(X_pca, y, test_size=0.1, random_state=30)
        svm_accuracy, svm_auc, svm_accuracy_pca, svm_auc_pca = svm_eval(X_train, y_train, X_test, y_test)
        # svclassifier = SVC(kernel='linear',probability=True)
        # svclassifier.fit(X_train, y_train)
        # y_pred = svclassifier.predict(X_test)
        # print('SVM accuracy:', svclassifier.score(X_test, y_test))
        # print('SVM classification report:\n', classification_report(y_test, y_pred))
        # y_scores = svclassifier.predict_proba(X_test)[:, 1]
        # fpr, tpr, thresholds = roc_curve(y_test, y_scores)
        # # Calculate AUC
        # svm_auc = auc(fpr, tpr)
        # print('SVM AUC value:', svm_auc)
        # newsvm=SVC(kernel='linear', probability=True)
        # newsvm.fit(X_train_pca, y_train)
        # y_pred = newsvm.predict(X_test_pca)
        # print('\nSVM accuracy for PCA:', newsvm.score(X_test_pca, y_test))
        # print('SVM classification report for PCA:\n', classification_report(y_test, y_pred))
        # y_scores = newsvm.predict_proba(X_test_pca)[:, 1]
        # fpr, tpr, thresholds = roc_curve(y_test, y_scores)
        # pcasvm_auc = auc(fpr, tpr)
        # print('SVM AUC value for PCA:', pcasvm_auc)
        knn_accuracy, knn_auc, knn_accuracy_pca, pca_knn = knn_eval(X_train, y_train, X_test, y_test)

        # knn = KNeighborsClassifier(n_neighbors=10)
        # knn.fit(X_train, y_train)
        # print('KNN accuracy:', knn.score(X_test, y_test))
        # y_pred = knn.predict(X_test)
        # print('KNN classification report:\n',classification_report(y_test, y_pred))
        # y_scores = knn.predict_proba(X_test)[:, 1]
        # fpr, tpr, thresholds = roc_curve(y_test, y_scores)
        # knn_auc = auc(fpr, tpr)
        # print('KNN AUC value:', knn_auc)
        # pknn = KNeighborsClassifier(n_neighbors=10)
        # pknn.fit(X_train_pca, y_train)
        # print('\nKNN accuracy for PCA:', pknn.score(X_test_pca, y_test))
        # y_pred = pknn.predict(X_test_pca)
        # print('KNN classification report for PCA\n', classification_report(y_test, y_pred))
        # y_scores = pknn.predict_proba(X_test_pca)[:, 1]
        # fpr, tpr, thresholds = roc_curve(y_test, y_scores)
        # pcaknn_auc = auc(fpr, tpr)
        # print('KNN AUC value for PCA',pcaknn_auc)
        rf_accuracy, rf_auc, rf_accuracy_pca, pcarf_auc = knn_eval(X_train, y_train, X_test, y_test)

        # model = RandomForestClassifier(n_estimators=40)
        # model.fit(X_train, y_train)
        # print('Random Forest accuracy:', model.score(X_test, y_test))
        # y_pred = model.predict(X_test)
        # print('Random Forest classification report\n',classification_report(y_test, y_pred))
        # y_scores = model.predict_proba(X_test)[:, 1]
        # fpr, tpr, thresholds = roc_curve(y_test, y_scores)
        # rf_auc = auc(fpr, tpr)
        # print('Random Forest AUC value:', rf_auc)
        # pmodel = RandomForestClassifier(n_estimators=40)
        # pmodel.fit(X_train_pca, y_train)
        # print('\nRandom Forest accuracy for PCA:', pmodel.score(X_test_pca, y_test))
        # y_pred = pmodel.predict(X_test_pca)
        # print('Random Forest classification report for PCA:\n',classification_report(y_test, y_pred))
        # y_scores = pmodel.predict_proba(X_test_pca)[:, 1]
        # fpr, tpr, thresholds = roc_curve(y_test, y_scores)
        # pcarf_auc = auc(fpr, tpr)
        # print('Random Forest AUC value for PCA:', pcarf_auc)
        lg_accuracy, lg_auc, lg_accuracy_pca, pcalg_auc = lg_eval(X_train, y_train, X_test, y_test)

        # lg = LogisticRegression()
        # lg.fit(X_train, y_train)
        # print('Logistic Regression accuracy:', lg.score(X_test, y_test))
        # y_pred = lg.predict(X_test)
        # print('Logistic Regression classification report:\n',classification_report(y_test, y_pred))
        # y_scores = lg.predict_proba(X_test)[:, 1]
        # fpr, tpr, thresholds = roc_curve(y_test, y_scores)
        # lg_auc = auc(fpr, tpr)
        # print('Logistic Regression AUC value:',lg_auc)
        # plg = LogisticRegression()
        # plg.fit(X_train_pca, y_train)
        # print('\nLogistic Regression accuracy for PCA:',plg.score(X_test_pca, y_test))
        # y_pred = plg.predict(X_test_pca)
        # print('Logistic Regression classification report for PCA:\n',classification_report(y_test, y_pred))
        # y_scores = plg.predict_proba(X_test_pca)[:, 1]
        # fpr, tpr, thresholds = roc_curve(y_test, y_scores)
        # pcalg_auc = auc(fpr, tpr)
        # print('Logistic Regression AUC value for PCA',pcalg_auc)
        xgb_accuracy, xgb_auc, xgb_accuracy_pca, pxgb_auc = xgb_eval(X_train, y_train, X_test, y_test)

        # clf = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=2)
        # clf.fit(X_train, y_train, eval_set=[(X_test, y_test)])
        # y_pred = clf.predict(X_test)
        # print('XGB accuracy:',clf.score(X_test, y_test))
        # print('XGB Classification report:\n',classification_report(y_test, y_pred))
        # y_scores = clf.predict_proba(X_test)[:, 1]
        # fpr, tpr, thresholds = roc_curve(y_test, y_scores)
        # xgb_auc = auc(fpr, tpr)
        # print('XGB AUC value:',xgb_auc)
        # pxgb = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=2)
        # pxgb.fit(X_train_pca, y_train, eval_set=[(X_test_pca, y_test)])
        # print('\nXGB accuracy for PCA:', pxgb.score(X_test_pca, y_test))
        # y_pred = pxgb.predict(X_test_pca)
        # print('XGB classification report for PCA:\n',classification_report(y_test, y_pred))
        # y_scores = pxgb.predict_proba(X_test_pca)[:, 1]
        # fpr, tpr, thresholds = roc_curve(y_test, y_scores)
        # pxgb_auc = auc(fpr, tpr)
        # print('XBG AUC for PCA:', pxgb_auc)
        model_row = [
            f"{disorder_name}_{label}_FC",
            svm_accuracy, svm_auc,
            knn_accuracy, knn_auc,
            rf_accuracy, rf_auc,
            lg_accuracy, lg_auc,
            xgb_accuracy, xgb_auc
        ]
        pca_row = [
            f"{disorder_name}_{label}_FCPCA",
            svm_accuracy_pca, pcasvm_auc,
            knn_accuracy_pca,pcaknn_auc,
            rf_accuracy_pca,pcarf_auc,
            lg_accuracy_pca,pcalg_auc,
            xgb_accuracy_pca, pxgb_auc
        ]
        all_model_data.append(model_row)
        all_model_data.append(pca_row)
        for row in all_model_data:
            print(row, type(row), len(row) if hasattr(row, '__len__') else 'Not iterable')
columns = ['Combination', 'SVM Accuracy', 'SVM AUC', 
           'KNN Accuracy', 'KNN AUC', 
           'Random Forest Accuracy','Random Forest AUC', 
           'Logistic Regression Accuracy', 'Logistic Regression AUC', 
           'XGB Accuracy', 'XGB AUC']
    
df = pd.DataFrame(all_model_data, columns=columns)
df.to_csv('../Data/Results/code.csv', mode='a',header=not os.path.exists('../Data/Results/code.csv'),index=False)